# 🤖 The ReAct Loop, Prepackaged: `create_agent`

## Learning Objectives
In this notebook, you will learn:
1. **`create_agent` is a ReAct graph** - What LangChain 1.x builds for you, and how to inspect it
2. **Multi-step and parallel tool use** - One agent chaining dependent calls, and firing independent calls in one turn
3. **Built-in and custom tools** - Loading a ready-made tool, and subclassing `BaseTool` for a retrieval tool
4. **Runaway loops and call limits** - Why an agent keeps searching, `ModelCallLimitMiddleware`, and fixing the real cause
5. **Conversation memory** - A checkpointer plus a `thread_id` turns a stateless agent into a multi-turn one

## Where this fits
`01_react_agent_langgraph.ipynb` built the ReAct loop by hand in LangGraph; `02_tool_calling_vs_react.ipynb` showed native tool calls replacing text parsing. LangChain 1.x's `create_agent` packages exactly that loop — native tool calls, `model ⇄ tools` — behind one function. The rest of this module applies the loop to research, coding and computer-use tasks; module 04 covers `create_agent`'s models, messages, structured output and middleware in depth.

## Prerequisites
- The ReAct loop — `01_react_agent_langgraph.ipynb`; defining tools — `01_defining_tools.ipynb` (module 02)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (used for the chat model and for embeddings in Part 3). All tools are simulated or built in this notebook, so no other keys are needed.

---
## 📦 Step 1: Environment Setup

Load the API key, then create the LLM every agent in this notebook shares.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

✅ Environment ready


In [2]:
# ============================================================================
# LLM INITIALIZATION: The model behind every agent below
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


---
## 🚀 Part 1: `create_agent` Is a ReAct Graph

**Agents** use an LLM as a **reasoning engine** to dynamically determine:
1. **Which actions** to perform
2. **In what order** to perform them
3. **When to stop** and return a final answer

`create_agent()` builds that as a compiled LangGraph graph. Every run follows the same loop:
1. **Model call**: the LLM sees the conversation plus the tool schemas
2. **Tool call**: if the model requests a tool, the graph executes it
3. **Observation**: the tool's result is appended as a `ToolMessage`
4. **Repeat** until the model replies without requesting a tool

> ⚠️ **A note on ReAct**: LangChain 0.x drove this loop with *prompt engineering* - the model was asked to emit `Thought: / Action: / Action Input:` text that LangChain then regex-parsed (you saw that in `02_tool_calling_vs_react.ipynb`). On LangChain 1.x, `create_agent()` uses the model's **native tool-calling API** instead, so there is no text to parse and nothing to fail parsing.

In [3]:
# ============================================================================
# TOOLS: A database search, a stock price lookup and an email sender (stubs)
# ============================================================================
from langchain_core.tools import tool


@tool
def search_database(query: str) -> str:
    """
    Search the internal database for information about a topic.
    Use this for internal company data, product information, or customer records.
    """
    # Simulated database search
    return f"Found 3 results for: {query}"


@tool
def get_stock_price(symbol: str) -> str:
    """
    Get the current stock price for a given stock symbol.

    Args:
        symbol: Stock ticker symbol (e.g., 'GOOGL', 'MSFT', 'AAPL')
    """
    # Simulated stock prices (in production, use a real API)
    prices = {"GOOGL": 175.50, "MSFT": 420.30, "AAPL": 195.80}
    price = prices.get(symbol.upper(), "Not found")
    return f"{symbol.upper()}: ${price}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """
    Send an email to the specified recipient.

    Args:
        to: Email address of the recipient
        subject: Subject line of the email
        body: Body content of the email
    """
    # Simulated email sending (in production, use SMTP or email API)
    return f"Email sent to {to} with subject: {subject}"


office_tools = [search_database, get_stock_price, send_email]

print(f"✅ Defined {len(office_tools)} tools: {[t.name for t in office_tools]}")

✅ Defined 3 tools: ['search_database', 'get_stock_price', 'send_email']


In [4]:
# ============================================================================
# CREATE THE AGENT and look at the graph it built
# ============================================================================
from langchain.agents import create_agent

office_agent = create_agent(
    llm,
    tools=office_tools,
    system_prompt=(
        "You are a helpful assistant. Use the available tools to help answer the "
        "user's questions. Always provide helpful and complete responses."
    ),
)

graph = office_agent.get_graph()
print("Returned type :", type(office_agent).__name__)
print("Graph nodes   :", list(graph.nodes))
print("Graph edges   :")
for edge in graph.edges:
    print(f"   {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

Returned type : CompiledStateGraph
Graph nodes   : ['__start__', 'model', 'tools', '__end__']
Graph edges   :
   __start__ -> model
   model -> __end__  (conditional)
   model -> tools  (conditional)
   tools -> model  (conditional)


Read the edges: `__start__ → model` is the entry point, `model → tools` is taken when the model requests a tool call, `tools → model` sends the result back for another model call (the ReAct feedback loop), and `model → __end__` is taken when the model answers without a tool call. It is the same graph you wired by hand in `01_react_agent_langgraph.ipynb`.

> **API note:** older material calls this `create_react_agent` from `langgraph.prebuilt` (deprecated since LangGraph v1.0), or builds the loop with `create_tool_calling_agent` + `AgentExecutor`. Two things changed: the import moved to `langchain.agents` and the name dropped "react", and the `prompt=` argument became `system_prompt=` — renaming only the function raises `TypeError`.

---
## 🔗 Part 2: Multi-Step and Parallel Tool Use

### 2.1 ⛓️ Dependent Calls

The task below cannot be done in one step: the email body needs the price, so the agent must call `get_stock_price`, *observe* the result, and only then call `send_email`. A small helper streams each new message so you can watch the loop — the 1.x replacement for `verbose=True`.

In [5]:
# ============================================================================
# HELPER: Stream an agent run and print each new message
# ============================================================================
def run_agent(agent, question, config=None):
    """Stream an agent run, pretty-printing each new message as it arrives."""
    final_state, last_id = None, None
    for state in agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
        stream_mode="values",
    ):
        message = state["messages"][-1]
        # Middleware nodes also emit state; print each message only once
        if message.id != last_id:
            message.pretty_print()
            last_id = message.id
        final_state = state
    return final_state


def model_calls(state) -> int:
    """How many times the model was called in a run (one AIMessage per call)."""
    return sum(1 for m in state["messages"] if m.type == "ai")


print("✅ Helpers ready")

✅ Helpers ready


In [6]:
# ============================================================================
# RUN: A two-step task where step 2 depends on step 1
# ============================================================================
result = run_agent(
    office_agent,
    "What's the stock price of Google? Once you get the stock price, send an email "
    "to analyst@example.com with the information.",
)
print(f"\n📞 Model calls: {model_calls(result)}")

================================ Human Message =================================

What's the stock price of Google? Once you get the stock price, send an email to analyst@example.com with the information.
================================== Ai Message ==================================
Tool Calls:
  get_stock_price (call_CtBIgwZxAPOY1X6V5TfQSD6F)
 Call ID: call_CtBIgwZxAPOY1X6V5TfQSD6F
  Args:
    symbol: GOOGL
================================= Tool Message =================================
Name: get_stock_price

GOOGL: $175.5
================================== Ai Message ==================================
Tool Calls:
  send_email (call_srqLr7Y2Q2wqDhMBWbvXHcnm)
 Call ID: call_srqLr7Y2Q2wqDhMBWbvXHcnm
  Args:
    to: analyst@example.com
    subject: Google Stock Price Update
    body: The current stock price of Google (GOOGL) is $175.5.
================================= Tool Message =================================
Name: send_email

Email sent to analyst@example.com with subject: Googl

### 2.2 ⚡ Independent Calls in Parallel

Many modern LLMs can call **multiple tools in parallel** when they're independent. `create_agent` handles this automatically: when one `AIMessage` carries several tool calls, the `tools` node runs them all before handing the results back to the model.

In [7]:
# ============================================================================
# PARALLEL TOOL CALLING: Weather for two cities and a calculation
# ============================================================================
@tool
def get_weather(city: str) -> str:
    """
    Get the current weather for a given city.

    Args:
        city: Name of the city (e.g., 'Bangalore', 'Mumbai', 'Delhi')
    """
    # This is simulated data - in production, you'd call a real weather API
    weather_data = {
        "bangalore": "28°C, Partly Cloudy",
        "mumbai": "32°C, Humid",
        "delhi": "25°C, Sunny",
    }
    return weather_data.get(city.lower(), "Weather data not available for this city")


@tool
def calculate(expression: str) -> str:
    """
    Evaluate a mathematical expression and return the result.

    Args:
        expression: A valid Python arithmetic expression (e.g., '25 * 4', '100 / 5 + 10')
    """
    try:
        return f"Result: {eval(expression, {'__builtins__': {}}, {})}"
    except Exception as e:
        return f"Error: {e}"


parallel_agent = create_agent(
    llm,
    tools=[get_weather, calculate],
    system_prompt="You are a helpful assistant. Call ALL tools needed to fully answer the user's request. When possible, call multiple tools in parallel.",
)

result = parallel_agent.invoke(
    {"messages": [("user", "Get weather in Bangalore and Mumbai, also calculate 25 * 4")]}
)

first_ai = next(m for m in result["messages"] if m.type == "ai")
print(f"🔧 Tool calls in the first model response: {len(first_ai.tool_calls)}")
for tc in first_ai.tool_calls:
    print(f"   - {tc['name']}({tc['args']})")
print(f"📞 Model calls: {model_calls(result)}")
print(f"\n📤 Final Output:\n{result['messages'][-1].content}")

🔧 Tool calls in the first model response: 3
   - get_weather({'city': 'Bangalore'})
   - get_weather({'city': 'Mumbai'})
   - calculate({'expression': '25 * 4'})
📞 Model calls: 2

📤 Final Output:
The current weather is as follows:
- **Bangalore**: 28°C, Partly Cloudy
- **Mumbai**: 32°C, Humid

Additionally, the result of the calculation \( 25 \times 4 \) is **100**.


Three tool calls, one model turn: the model saw that none of them depended on another. Compare with 2.1, where the dependency forced two turns. The loop is the same; the model decides how much to do per turn.

---
## 🧰 Part 3: Built-in and Custom Tools

| Method | Complexity | Use Case |
|--------|------------|----------|
| Built-in tools (`load_tools`) | None | Common utilities provided by `langchain_community` |
| `@tool` decorator | Simple | Quick functions with no state |
| `BaseTool` subclass | Medium | When you need more control (configuration, state, async) |

> 💡 **Pro Tip**: The `description` is extremely important! The agent uses it to decide which tool to use. Make it clear and specific.

### 3.1 🧮 A Built-in Tool

`llm-math` is a calculator that itself uses an LLM to turn a word problem into an expression. Note the 1.x import: `load_tools` lives in `langchain_community` now.

In [8]:
# ============================================================================
# BUILT-IN TOOL: llm-math, plus a call cap
# ============================================================================
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_community.agent_toolkits.load_tools import load_tools

# Some tools need an LLM of their own (llm-math turns words into an expression)
math_tools = load_tools(["llm-math"], llm=llm)
print("Loaded:", [(t.name, t.description[:60]) for t in math_tools])

math_agent = create_agent(
    llm,
    math_tools,
    system_prompt="You are a helpful assistant. Use the tools available to you whenever they help answer the question.",
    # run_limit caps model calls per invocation; exit_behavior="end" returns what
    # the agent has so far instead of raising. (This replaces 0.x's max_iterations.)
    middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")],
)

Loaded: [('Calculator', 'Useful for when you need to answer questions about math.')]


In [9]:
# ============================================================================
# RUN: One question the model knows, one that needs the calculator
# ============================================================================
print("=" * 30, "General knowledge", "=" * 30)
result = run_agent(math_agent, "How many members does the A-Team have?")
print(f"📞 Model calls: {model_calls(result)} (no tool needed)\n")

print("=" * 30, "Math", "=" * 30)
result = run_agent(math_agent, "What is 100 devided by 25?")  # the typo is deliberate
print(f"📞 Model calls: {model_calls(result)} (model → Calculator → model)")

============================== General knowledge ==============================
================================ Human Message =================================

How many members does the A-Team have?
================================== Ai Message ==================================

The A-Team, a popular television series from the 1980s, features a group of four main members:

1. John "Hannibal" Smith - the leader and strategist.
2. Templeton "Faceman" Peck - the conman and smooth talker.
3. B.A. Baracus - the mechanic and strongman.
4. H.M. "Howling Mad" Murdock - the pilot and eccentric member.

So, the A-Team has four main members.
📞 Model calls: 1 (no tool needed)

============================== Math ==============================
================================ Human Message =================================

What is 100 devided by 25?
================================== Ai Message ==================================
Tool Calls:
  Calculator (call_Vqd2KWGJFZbaowK5rqSGnVcG)
 Call ID:

The first question ends after one model call: the agent recognizes the tool won't help. The second goes model → `Calculator` → model — and even with the typo "devided", the agent understands the intent.

### 3.2 🍝 A Custom Retrieval Tool with `BaseTool`

Real applications need domain-specific tools. Here the domain is a restaurant, *Bella Vista*, and the tool searches its FAQ in a small vector store built right here from inline data (FAISS in memory, OpenAI embeddings). We build **two** stores from the same FAQ, because Part 4 needs both:

- `fragmented_store` — a bad ingestion: only the **questions** made it into the index, the answers were lost
- `faq_store` — each question and its answer stored together, as one document

In [10]:
# ============================================================================
# KNOWLEDGE BASE: The Bella Vista FAQ, indexed two ways
# ============================================================================
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

FAQ = [
    ("What are your opening hours?", "We open at 11:30 AM and close at 10:00 PM, Tuesday to Sunday. We are closed on Mondays."),
    ("Do you take reservations?", "Yes. Book online or call (555) 010-2233; we hold tables for 15 minutes."),
    ("Is there a vegetarian menu?", "Yes, a full vegetarian menu, and most pasta dishes can be made vegan."),
    ("Do you have gluten-free options?", "Gluten-free pasta and pizza bases are available on request."),
    ("Is there parking?", "Free parking for guests is behind the building, entrance on Elm Street."),
    ("Can I bring my own wine?", "Yes, corkage is $15 per bottle."),
    ("Do you cater private events?", "Our private room seats up to 30 guests; email events@bellavista.example."),
    ("What is the dress code?", "Smart casual. No dress code is enforced at lunch."),
    ("Are children welcome?", "Children are welcome; we have high chairs and a kids' menu."),
    ("Is there outdoor seating?", "Our terrace seats 20 and is open from May to September."),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Bad ingestion: questions only, answers lost
fragmented_store = FAISS.from_texts([q for q, _ in FAQ], embeddings)
# Good ingestion: each question stored with its answer
faq_store = FAISS.from_texts([f"Q: {q}\nA: {a}" for q, a in FAQ], embeddings)

print(f"✅ Indexed {len(FAQ)} FAQ entries twice (fragmented and complete)")

✅ Indexed 10 FAQ entries twice (fragmented and complete)


In [11]:
# ============================================================================
# CUSTOM TOOL: Subclass BaseTool
# ============================================================================
# NOTE (LangChain 1.x): the callback managers moved to langchain_core.
# `from langchain.callbacks.manager import ...` raises ModuleNotFoundError on 1.x.
from typing import Any, Optional

from langchain_core.callbacks.manager import CallbackManagerForToolRun
from langchain_core.tools import BaseTool


class CustomSearchTool(BaseTool):
    """A custom tool that searches the restaurant FAQ vector store."""

    # REQUIRED: Tool name - keep it identifier-friendly (some providers reject spaces)
    name: str = "restaurant_search"

    # REQUIRED: Description - the agent uses this to decide when to use the tool
    description: str = "useful for when you need to answer questions about our restaurant"

    # Configuration: which vector store to search (a field, so each instance can differ)
    store: Any

    def _run(self, query: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        """Called when the agent uses this tool: semantic search, then join the hits."""
        docs = self.store.as_retriever().invoke(query)
        return "\n".join(doc.page_content for doc in docs)


# The agent below uses the FRAGMENTED store, as the original notebook's data did
restaurant_agent = create_agent(
    llm,
    [CustomSearchTool(store=fragmented_store)],
    system_prompt=(
        "You are a helpful assistant for the Bella Vista restaurant. "
        "Use the restaurant_search tool for anything about the restaurant. "
        "For unrelated questions, answer from your own knowledge."
    ),
    middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")],
)

print("Test 1: a question outside the tool's scope")
result = run_agent(restaurant_agent, "Who are the members of the A-Team?")
print(f"📞 Model calls: {model_calls(result)}")

Test 1: a question outside the tool's scope
================================ Human Message =================================

Who are the members of the A-Team?
================================== Ai Message ==================================

The A-Team is a fictional group of characters from the 1980s television series of the same name. The main members of the A-Team are:

1. **Colonel John "Hannibal" Smith** - The leader of the team, known for his tactical genius and love of disguises.
2. **Lieutenant Templeton "Faceman" Peck** - The team's con artist and smooth-talker, skilled in acquiring necessary items and resources.
3. **Sergeant Bosco "B.A." Baracus** - The team's mechanic and strongman, known for his toughness and fear of flying.
4. **Captain H.M. "Howling Mad" Murdock** - The team's pilot, who is often portrayed as being mentally unstable but is actually quite skilled.

These characters work together to help those in need while evading the law, as they are wrongly accused of 

The agent only has the restaurant tool, but this question is about a TV show, so it answers from its own knowledge without wasting a search. Now a question the tool *should* answer.

---
## 🔄 Part 4: Runaway Loops and Call Limits

On 0.x this surfaced as `"Agent stopped due to iteration limit"`. On 1.x, `ModelCallLimitMiddleware` ends the run at the cap instead, so you get a truncated answer rather than an error string - but the underlying cause is the same.

**Root cause**: the vector store returns fragmented results (questions without answers), so the model keeps calling the tool hoping for something better — or gives up without an answer.

In [12]:
# ============================================================================
# TEST 2: An in-scope question against the fragmented store
# ============================================================================
result = run_agent(restaurant_agent, "When does the restaurant open?")
print(f"\n📞 Model calls: {model_calls(result)} (cap: 3)")

================================ Human Message =================================

When does the restaurant open?
================================== Ai Message ==================================
Tool Calls:
  restaurant_search (call_meTc8e4ReYzXQZdeglAFxavQ)
 Call ID: call_meTc8e4ReYzXQZdeglAFxavQ
  Args:
    query: opening hours
================================= Tool Message =================================
Name: restaurant_search

What are your opening hours?
Do you take reservations?
Is there outdoor seating?
Are children welcome?
================================== Ai Message ==================================

I couldn't find the specific opening hours for Bella Vista. However, you can typically find this information on the restaurant's official website or by calling them directly. Would you like to know anything else about the restaurant?

📞 Model calls: 2 (cap: 3)


Look at the `ToolMessage`: the search "works" — it returns the most similar entries — but they are all *questions*. There is no opening time anywhere in what the agent can see. This agent was allowed to say so, and it did: the user gets no answer.

An agent told to *persist* shows the other failure mode. Same tool, same data, but the system prompt insists the answer is in there:

In [13]:
# ============================================================================
# TEST 3: A persistent agent on the fragmented store — the cap has to stop it
# ============================================================================
persistent_agent = create_agent(
    llm,
    [CustomSearchTool(store=fragmented_store)],
    system_prompt=(
        "You are a helpful assistant for the Bella Vista restaurant. The answer to every "
        "restaurant question is in the knowledge base: keep searching with different "
        "queries until you find it. Never tell the user you could not find something."
    ),
    middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")],
)

result = run_agent(persistent_agent, "When does the restaurant open?")
searches = sum(1 for m in result["messages"] if m.type == "tool")
print(f"\n🔎 Searches before the run was stopped: {searches}")
print(f"🏁 What the user gets: {result['messages'][-1].content!r}")

================================ Human Message =================================

When does the restaurant open?
================================== Ai Message ==================================
Tool Calls:
  restaurant_search (call_sFo3femTUBEtVVdOrHHsvgo5)
 Call ID: call_sFo3femTUBEtVVdOrHHsvgo5
  Args:
    query: restaurant opening hours
================================= Tool Message =================================
Name: restaurant_search

What are your opening hours?
Do you take reservations?
Is there outdoor seating?
Is there a vegetarian menu?
================================== Ai Message ==================================
Tool Calls:
  restaurant_search (call_sTgm3DiJTb41srAJUPzzPUBi)
 Call ID: call_sTgm3DiJTb41srAJUPzzPUBi
  Args:
    query: opening hours
================================= Tool Message =================================
Name: restaurant_search

What are your opening hours?
Do you take reservations?
Is there outdoor seating?
Are children welcome?
================

Now the agent searches again and again with reworded queries, each returning the same questions without answers, until `ModelCallLimitMiddleware` ends the run at 3 model calls and appends its own notice as the last message. The cap did its job — the bill is bounded — but the user still has no opening hours.

**Fixes (in order of effectiveness):**

| Fix | Where | Code Change |
|-----|-------|-------------|
| 1. Fix the data | ingestion | store each question *with* its answer |
| 2. Retrieve more docs | `as_retriever()` | `search_kwargs={"k": 6}` |
| 3. Better formatting | `_run()` return | Add clear headers like "Restaurant Information:" |
| 4. Improve description | `description` field | Be specific about what the tool returns |
| 5. Raise the call cap | `create_agent()` | `ModelCallLimitMiddleware(run_limit=5)` |

Fix 5 is the *least* effective - a higher cap buys more attempts at a retrieval problem rather than solving it. Fixes 1–4 change what the model actually sees. The improved tool below applies all of them.

In [14]:
# ============================================================================
# 🔧 FIX: Improved tool over the complete FAQ store
# ============================================================================
class ImprovedSearchTool(BaseTool):
    """Improved restaurant search tool with better data, retrieval and formatting."""

    name: str = "restaurant_search"

    # FIX 4: More descriptive - tells the agent what to expect
    description: str = (
        "Search the Bella Vista restaurant knowledge base. "
        "Use for: hours, menu, reservations, ambiance, policies. "
        "Returns FAQ answers - extract the information needed to answer the user."
    )
    store: Any

    def _run(self, query: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        # FIX 2: Retrieve more documents (6 instead of the default 4)
        docs = self.store.as_retriever(search_kwargs={"k": 6}).invoke(query)
        if not docs:
            return "No information found."
        # FIX 3: Format results clearly so the model can use them directly
        return "Restaurant Information:\n" + "\n".join(f"• {doc.page_content}" for doc in docs)


improved_agent = create_agent(
    llm,
    # FIX 1: the complete store, questions with their answers
    [ImprovedSearchTool(store=faq_store)],
    system_prompt=(
        "You are a helpful assistant for the Bella Vista restaurant. "
        "Search the knowledge base once, then answer from what you find. "
        "Do not repeat the same search."
    ),
    middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")],
)

result = run_agent(improved_agent, "When does the restaurant open?")
print(f"\n📞 Model calls: {model_calls(result)} (cap: 3, unchanged)")

================================ Human Message =================================

When does the restaurant open?
================================== Ai Message ==================================
Tool Calls:
  restaurant_search (call_3ZBP1uwUi8Et24eM0s5QNz36)
 Call ID: call_3ZBP1uwUi8Et24eM0s5QNz36
  Args:
    query: opening hours
================================= Tool Message =================================
Name: restaurant_search

Restaurant Information:
• Q: What are your opening hours?
A: We open at 11:30 AM and close at 10:00 PM, Tuesday to Sunday. We are closed on Mondays.
• Q: Do you take reservations?
A: Yes. Book online or call (555) 010-2233; we hold tables for 15 minutes.
• Q: Is there outdoor seating?
A: Our terrace seats 20 and is open from May to September.
• Q: Are children welcome?
A: Children are welcome; we have high chairs and a kids' menu.
• Q: What is the dress code?
A: Smart casual. No dress code is enforced at lunch.
• Q: Do you cater private events?
A: Our priva

Same question, same cap — and now one search is enough: the `ToolMessage` contains the answer, so the model's second call writes it. We did not raise the limit; we changed what the model sees.

---
## 🧠 Part 5: Conversation Memory with a Checkpointer

The agents so far are **stateless** - each `invoke()` starts from an empty message list. For chatbot-like applications, we need agents that remember the conversation history, can refer back to previous topics, and maintain context across multiple turns.

On LangChain 1.x, the 0.x `ConversationBufferMemory` class is gone (`langchain.memory` no longer exists). Persistence is a graph concern: you pass a **checkpointer**, and every run tagged with the same `thread_id` resumes the same conversation.

| Feature | Stateless Agent | Conversational Agent |
|---------|-----------------|---------------------|
| Memory | none | `checkpointer=InMemorySaver()` |
| Context | single query | full thread, restored automatically |
| Selector | n/a | `config={"configurable": {"thread_id": "..."}}` |
| Use Case | one-off questions | multi-turn chat |

In [15]:
# ============================================================================
# CONVERSATIONAL AGENT: Same create_agent call, plus a checkpointer
# ============================================================================
from langgraph.checkpoint.memory import InMemorySaver

# The checkpointer stores the message list after every step.
# InMemorySaver is process-local; swap in a SQLite/Postgres saver to persist.
chat_agent = create_agent(
    llm,
    [ImprovedSearchTool(store=faq_store)],
    system_prompt="You are a helpful conversational assistant for the Bella Vista restaurant.",
    checkpointer=InMemorySaver(),
)

# Every call that passes this config continues the SAME conversation
config = {"configurable": {"thread_id": "demo-conversation-1"}}

state = chat_agent.get_state(config)
print(f"Messages stored on thread {config['configurable']['thread_id']!r}: "
      f"{len(state.values.get('messages', []))}")

Messages stored on thread 'demo-conversation-1': 0


In [16]:
# ============================================================================
# THREE TURNS ON ONE THREAD: "there" and "what about" only make sense with memory
# ============================================================================
for question in [
    "What is the capital of France?",
    "Any suggestions what to visit there?",
    "What about China?",
]:
    print("=" * 80)
    result = chat_agent.invoke({"messages": [("user", question)]}, config=config)
    print(f"🧑 {question}\n🤖 {result['messages'][-1].content[:400]}")
    print(f"   (messages on the thread: {len(result['messages'])})")

🧑 What is the capital of France?
🤖 The capital of France is Paris.
   (messages on the thread: 2)
🧑 Any suggestions what to visit there?
🤖 In Paris, you can visit several iconic attractions, including:

1. **Eiffel Tower** - A must-see landmark offering stunning views of the city.
2. **Louvre Museum** - Home to thousands of works of art, including the Mona Lisa.
3. **Notre-Dame Cathedral** - A masterpiece of French Gothic architecture.
4. **Montmartre** - A charming neighborhood known for its artistic history and the Basilica of Sacr
   (messages on the thread: 4)
🧑 What about China?
🤖 In China, there are numerous incredible places to visit, including:

1. **Great Wall of China** - One of the most famous landmarks in the world, stretching over 13,000 miles.
2. **Forbidden City (Beijing)** - A vast imperial palace complex with stunning architecture and rich history.
3. **Terracotta Army (Xi'an)** - An archaeological site featuring thousands of life-sized clay soldiers.
4. **Li Ri
   (me

In [17]:
# ============================================================================
# A DIFFERENT thread_id starts from scratch
# ============================================================================
fresh = chat_agent.invoke(
    {"messages": [("user", "What about China?")]},
    config={"configurable": {"thread_id": "demo-conversation-2"}},
)
print(f"🤖 (new thread) {fresh['messages'][-1].content[:400]}")
print(f"   (messages on the new thread: {len(fresh['messages'])})")

🤖 (new thread) It seems like you're asking about the Bella Vista restaurant, which has various policies and offerings. Here are some key points:

- **Children**: They are welcome, and the restaurant provides high chairs and a kids' menu.
- **Reservations**: You can make reservations online or by calling (555) 010-2233. They hold tables for 15 minutes.
- **Dress Code**: The dress code is smart casual, but no spec
   (messages on the new thread: 4)


On the first thread, "there" resolved to Paris and "What about China?" was read as "what to visit in China" — the model saw the earlier turns because the checkpointer replayed them. On the new thread the same words have no context, and the agent has to guess or ask. This is strictly more capable than the old `Memory` objects: swap `InMemorySaver` for a Postgres or SQLite checkpointer and the same code persists across process restarts.

### 🎯 Try It Yourself
1. Give `parallel_agent`'s tools plus `ImprovedSearchTool` to one agent and ask a question that needs both.
2. Hold a multi-turn conversation on two `thread_id`s and confirm they stay separate.
3. Add `SummarizationMiddleware` alongside `ModelCallLimitMiddleware` and watch it compress a long conversation (module 04, `04_middleware.ipynb`, shows how).

---
## 📝 Summary

In this notebook, we learned:

### 1. `create_agent` Is a ReAct Graph
- It returns a compiled LangGraph graph: `model ⇄ tools`, ending when the model stops calling tools
- `create_react_agent(prompt=...)` and `AgentExecutor` are its predecessors; the 1.x call is `create_agent(llm, tools, system_prompt=...)`

### 2. Multi-Step and Parallel Tool Use
- Dependent calls take one model turn each; independent calls can share a turn
- A streaming helper replaces 0.x's `verbose=True`

### 3. Built-in and Custom Tools
- `load_tools` (now in `langchain_community`) gives ready-made tools; `BaseTool` subclasses carry configuration such as which store to search

### 4. Runaway Loops and Call Limits
- `ModelCallLimitMiddleware` bounds a run (`run_limit` per invocation, `thread_limit` per conversation)
- A cap stops the damage; fixing what the tool returns removes the cause

### 5. Conversation Memory
- `checkpointer=` plus a `thread_id` in the config gives multi-turn memory; different threads are independent

| LangChain 0.x | LangChain 1.x |
|---------------|---------------|
| `initialize_agent(tools, llm, agent=AgentType...)` | `create_agent(llm, tools, system_prompt=...)` |
| `max_iterations=3` | `ModelCallLimitMiddleware(run_limit=3)` |
| `verbose=True` | `.stream(..., stream_mode="values")` |
| `memory=ConversationBufferMemory(...)` | `checkpointer=` + `thread_id` in config |
| `invoke({"input": ...})` -> `{"output": ...}` | `invoke({"messages": [...]})` -> `{"messages": [...]}` |

### Next Steps
- **`04_web_research_agent.ipynb`**: a research task done two ways — a fixed plan → search → report pipeline, and a ReAct agent